# Homework 3 - Functions 1
#### Natalie Masetti

### Preparation

In [1]:
import pandas as pd
import numpy as np
import os
os.getcwd()

os.chdir(r"C:/Users/natal/OneDrive/Documents/GitHub/QMSS-GR5072-Spring2026-HWs/homework-2-main") # desktop

In [2]:
path = r"C:/Users/natal/OneDrive/Documents/GitHub/QMSS-GR5072-Spring2026-HWs/homework-3-main/Data"

In [3]:
# Load data in
flights = pd.read_csv(path + r"/nycflights.csv")

### 1: Write a function that takes a single numerical variable and returns three values, the minimum number, the median, and the maximum number of the vector. Test your function using the month column of the flights data set.

In [4]:
def col_stats(col):
    print('This is the minimum:', col.min())
    print('This is the median:', col.median())
    print('This is the maximum:', col.max())

In [5]:
col_stats(flights['month'])

This is the minimum: 1
This is the median: 7.0
This is the maximum: 12


## 2: Explain your reasoning for choosing your function's name in the previous question.

I chose "col_stats" as my function name because it is short and explains what the function does (gives descriptive statistics for a column). It doesn't have a verb, but I think it's concise and clear as is. It's also in snake case. 

In [6]:
flights['dep_time'].describe()

count    328521.000000
mean       1349.109947
std         488.281791
min           1.000000
25%         907.000000
50%        1401.000000
75%        1744.000000
max        2400.000000
Name: dep_time, dtype: float64

## 3: Write a function that categorizes a numerical variable in the flights data into four categories.

The function should have two arguments. The first should represent the data object and the second should represent a column name in the data object.

The function should return one new column which categorizes the dep_time column into four categories in the following manner. For any particular variable in the flights data that represents military time (i.e., 0 to 2400 where 1200 represents 12 in the afternoon and 2400 represents midnight), the function should classify values into four categories:

"Morning" for values from 5 am to 11:59 am

"Afternoon" for values from 12 pm to 4:59 pm

"Evening" for values from 5 pm to 8:59 pm

"Night" for values from 9 pm to 4:59 am

In [7]:
def time_categorizer(df, col):
    other = df.copy()
    other[col] = other[col].replace(range(500,1200), "morning")
    other[col] = other[col].replace(range(1200,1800), "afternoon")
    other[col] = other[col].replace(range(1800,2100), "evening")
    other[col] = other[col].replace(range(2100,2400), "night")
    other[col] = other[col].replace(range(0000,500), "night")
    other.rename(columns={col:'category'}, inplace=True)
    other['time_category'] = other['category'].copy()
    df = df.merge(other)
    return df

In [8]:
flights = time_categorizer(flights, 'dep_time')

In [9]:
flights['time_category'].value_counts()

time_category
morning      129539
afternoon    120761
evening       57649
night         20543
2400.0           29
Name: count, dtype: int64

## 4: Explain your reasoning for choosing your function's name in the previous question.

I chose "time_categorizer" for my function name because, although it is a little long, it is a snake-case, clear descriptor of what the function does. It's also in the same general format of my earlier function: what it's working on (time, col) + what it does/outputs (stats, categorizes).  

## 5: Write a function that calculates the median of all numeric variables in the flights dataset.

Hint: There are several ways to subset a DF to numeric values only, you will need to search online (using google, stackexchange or some generative AI like chat GPT) to find a suitable way to do so. Alternatively, you could try to do this manually with a for loop, which was reviewed in datacamp, although as a class we are not learning for loops together until next week.

In [10]:
def col_medians(df):
    numeric_only = df.select_dtypes((int, float))
    return(numeric_only.median())

# used https://stackoverflow.com/questions/73720788/how-to-get-only-numeric-type-columns-from-dataframe for returning numeric only columns

In [11]:
col_medians(flights)

Unnamed: 0        168388.5
year                2013.0
month                  7.0
day                   16.0
dep_time            1401.0
sched_dep_time      1359.0
dep_delay             -2.0
arr_time            1535.0
sched_arr_time      1556.0
arr_delay             -5.0
flight              1496.0
air_time             129.0
distance             872.0
hour                  13.0
minute                29.0
dtype: float64

## 6: Explain your reasoning for choosing your function's name in the previous question.

I named my function "col_medians" because it is also a snake-case, concise descriptor that's, again, in the same format of my earlier functions: what it's working on (col) + what it does/outputs (median).  Notably it is not just "median" to avoid overwriting the median function.

## 7: Modify the function t_test() we wrote in class together this week so that this function can handle violations to the homogeneity of variance (HOV) assumption. This assumption is described below in case you are not familiar with it. Please read the class activity carefully - as all of those details are relevant.

Create temporary objects within your function to hold the sample means, variances, and sample sizes, objects which will be used in the computation for both versions of the t-test (i.e., you should create these objects once, before you jump into an if-else statement about whether the HOV assumption is broken)

Compare the sample variances. Use an if-else statement that depends on the ratio of the sample variances. If this ratio is in between 0.25 - 4 (inclusive of the 0.25 and 4.0), then:

Conduct the standard t test, using the code from the class activity. The last column of the return output, "test", should now read "Independent samples t-test" since we need to be more specific here to distinguish this type of t test from Welch's t-test

Also, print the following message: "The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted"

If the HOV assumption is violated, then:

Conduct Welch's t-test, using the formulas above

Return an object with the same 9 columns from the class activity, but the "test" column should read "Welch's t-test"

Also, print the following message: "The homogeneity of variance assumption was violated, so Welch's t-test was conducted"

In [22]:
import scipy.stats as stats
# Create t-test function
def t_test(num_var, bin_var):

    # holding sample means, variances, and sample sizes
    group1 = num_var[bin_var == 0]
    group2 = num_var[bin_var == 1]
    group1_mean = group1.mean()
    group2_mean = group2.mean()
    s2_1 = group1.var()
    s2_2 = group2.var()
    n1 = group1.shape[0]
    n2 = group2.shape[0]

    if (s2_1/s2_2) >= 0.25 or (s2_1/s2_2) <= 4:
        mean_diff = group2.mean() - group1.mean()
        
        # Calculate the degrees of freedom (DF)
        DF = n1 + n2 - 2
        
        # Calculate the pooled standard deviation (sp)
        sp = np.sqrt(((n1 - 1) * s2_1 + (n2 - 1) * s2_2) / DF)
        
        # Calculate the standard error of the mean differenc
        SE_mean_diff = sp * np.sqrt(1/n1 + 1/n2)
        
        # Calculate the t-statistic
        t_statistic = mean_diff / SE_mean_diff
        
        # Calculate the p-value
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), DF))

        print('The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted')

        # Put the results into a DataFrame
        results = pd.DataFrame({
            "Continuous Variable": [num_var.name],
            "Binary Variable": [bin_var.name],
            "Total Sample Size": [n1 + n2],
            "Mean Difference": [round(mean_diff, 2)], 
            "SE of Mean Difference": [round(SE_mean_diff, 2)],
            "DF": [DF],
            "t-statistic": [round(t_statistic, 3)],
            "P-value": [("%.3f" % p_value).lstrip('0')],
            "Test": "Independent samples t-test"
        })
        
        return results

    else:
        mean_diff = group2.mean() - group1.mean()
        
        # Calculate the degrees of freedom (DF)
        DF = (((((s2_1**2) / n1) + ((s2_2**2) / n2)))**2) / ((1 / n1-1)*(((s2_1**2)/n1)**2)) + ((1 / n2-1)*(((s2_2**2)/n2)**2))
        
        # Calculate the pooled standard deviation (sp)
        sp = np.sqrt(((n1 - 1) * s2_1 + (n2 - 1) * s2_2) / DF)
        
        # Calculate the standard error of the mean differenc
        SE_mean_diff = sp * np.sqrt(1/n1 + 1/n2)
        
        # Calculate the t-statistic
        t_statistic = (group1.mean() - group2.mean()) / (math.sqrt(((s2_1**2) / n1) + ((s2_2**2) / n2)))
        
        # Calculate the p-value
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), DF))

    
        print('The homogeneity of variance assumption was violated, so Welchs t-test was conducted')
        
        # Put the results into a DataFrame
        results = pd.DataFrame({
            "Continuous Variable": [num_var.name],
            "Binary Variable": [bin_var.name],
            "Total Sample Size": [n1 + n2],
            "Mean Difference": [round(mean_diff, 2)], 
            "SE of Mean Difference": [round(SE_mean_diff, 2)],
            "DF": [DF],
            "t-statistic": [round(t_statistic, 3)],
            "P-value": [("%.3f" % p_value).lstrip('0')],
            "Test": "Welch's t-test"
        })
        
        return results

## #8: Import the GED data set we used for the class activity. Call the t_test() function and test it out on the GED data set we used in class. Let the numeric variable be 'income_log' and let the binary variable be 'ged'.

In [20]:
path = r"C:\Users\natal\OneDrive\Documents\GitHub\QMSS-GR5072-Spring2026\Week 5\Activity"
data = pd.read_csv(path + r"\3 ged_data.csv")  

In [23]:
t_test(data['income_log'], data['ged'])

The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted


,Continuous Variable,Binary Variable,Total Sample Size,Mean Difference,SE of Mean Difference,DF,t-statistic,P-value,Test
0,income_log,ged,5976,-0.48,0.06,5974,-7.458,.000,Independent samples t-test


## Bibliography
- ThePyGuy. (2022, September 14). *Answer to: How to get only numeric type columns from dataframe \[duplicate\]* \[Source code\]. https://stackoverflow.com/questions/73720788/how-to-get-only-numeric-type-columns-from-dataframe